In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

ZIP_PATH = "/content/drive/MyDrive/dl-fa26-resnet-project/data.zip"
EXTRACT_DIR = "/content/dataset"

# 3. Extract to Colab's local high-speed storage
if not os.path.exists(EXTRACT_DIR):
    print("Extracting dataset... This will take a minute or two.")
    !unzip -q "{ZIP_PATH}" -d "{EXTRACT_DIR}"
    print("Extraction complete!")
else:
    print("Dataset already extracted.")

# 4. Automatically find the exact folder containing the CSVs inside the unzipped data
BASE_DIR = EXTRACT_DIR
for root, dirs, files in os.walk(EXTRACT_DIR):
    if 'train_labels.csv' in files:
        BASE_DIR = root
        break

print(f"\nDataset is ready at: {BASE_DIR}")

Extracting dataset... This will take a minute or two.
Extraction complete!

Dataset is ready at: /content/dataset


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Turns on automatic hardware optimization for static image sizes
torch.backends.cudnn.benchmark = True

# ==========================================
# 1. ARCHITECTURE (<5M Params)
# ==========================================
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class CustomBasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, use_se=True):
        super(CustomBasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.use_se = use_se
        if self.use_se:
            self.se = SEBlock(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.use_se:
            out = self.se(out)
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class EfficientResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(EfficientResNet, self).__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.linear = nn.Linear(256 * block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

def get_efficient_resnet():
    return EfficientResNet(CustomBasicBlock, [4, 4, 3])


# ==========================================
# 2. DATASET READER (RAM PRELOADER)
# ==========================================
class KaggleCIFARDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None, is_test=False):
        self.df = pd.read_csv(csv_file)
        self.df.columns = [str(c).strip().lower() for c in self.df.columns]
        self.transform = transform
        self.is_test = is_test

        self.images = []
        self.targets = []

        print(f"\nPreloading {'Test' if is_test else 'Train'} images into RAM. This takes ~1 minute...")
        for idx in tqdm(range(len(self.df)), desc="Loading"):
            row = self.df.iloc[idx]

            raw_id = str(row['id']) if 'id' in self.df.columns else str(row.iloc[0])
            clean_id_number = ''.join(filter(str.isdigit, raw_id))

            img_name = f"test_{clean_id_number.zfill(4)}.png" if self.is_test else f"train_{clean_id_number.zfill(5)}.png"
            img_path = os.path.join(img_dir, img_name)

            if not self.is_test:
                target = int(row['label']) if 'label' in self.df.columns else int(row.iloc[1])
            else:
                target = row['id'] if 'id' in self.df.columns else raw_id

            image = Image.open(img_path).convert("RGB")
            self.images.append(image)
            self.targets.append(target)

        print("Preloading complete!")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image = self.images[idx]
        target = self.targets[idx]

        if self.transform:
            image = self.transform(image)

        return (target, image) if self.is_test else (image, target)


# Transforms with AutoAugment for maximum accuracy
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Directories (Uses the BASE_DIR found automatically in Cell 1)
train_csv_path = os.path.join(BASE_DIR, "train_labels.csv")
train_img_dir = os.path.join(BASE_DIR, "train_images")
test_csv_path = os.path.join(BASE_DIR, "sample_submission.csv")
test_img_dir = os.path.join(BASE_DIR, "test_images")

# Datasets & Loaders
trainset = KaggleCIFARDataset(csv_file=train_csv_path, img_dir=train_img_dir, transform=transform_train, is_test=False)
testset = KaggleCIFARDataset(csv_file=test_csv_path, img_dir=test_img_dir, transform=transform_test, is_test=True)

trainloader = DataLoader(trainset, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
testloader = DataLoader(testset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)


# ==========================================
# 3. TRAINING LOOP
# ==========================================
def train(model, dataloader, epochs, lr):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):
        model.train()
        train_loss, correct, total = 0, 0, 0

        pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Epoch {epoch+1}/{epochs}")
        for batch_idx, (inputs, targets) in pbar:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            pbar.set_postfix({"Loss": f"{train_loss/(batch_idx+1):.3f}", "Acc": f"{100.0 * correct / total:.2f}%"})

        scheduler.step()


# ==========================================
# 4. GENERATE KAGGLE SUBMISSION
# ==========================================
def create_kaggle_submission(model, dataloader, filename="submission.csv"):
    model.eval()
    print("\nGenerating predictions for hidden test set...")

    results = []
    with torch.no_grad():
        for ids, inputs in tqdm(dataloader, desc="Predicting"):
            inputs = inputs.to(device)
            outputs = model(inputs)
            predictions = outputs.argmax(dim=1).cpu().numpy()

            for img_id, pred in zip(ids, predictions):
                if isinstance(img_id, torch.Tensor):
                    img_id = img_id.item()
                results.append({"id": img_id, "label": pred})

    df = pd.DataFrame(results)
    df.to_csv(filename, index=False)
    print(f"Saved {len(df)} predictions to {filename}. Ready to submit!")


# ==========================================
# 5. RUN EVERYTHING
# ==========================================
model = get_efficient_resnet()
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal Trainable Parameters: {total_params:,}\n")

# FULL RUN: 120 Epochs to guarantee maximum convergence
EPOCHS = 120
train(model, trainloader, epochs=EPOCHS, lr=0.1)

# Save directly to Google Drive!
drive_out_path = "/content/drive/MyDrive/dl-fa26-resnet-project/"

# Save Model Weights
model_save_path = os.path.join(drive_out_path, "efficient_resnet_final.pth")
torch.save(model.state_dict(), model_save_path)
print(f"\nModel saved safely to {model_save_path}")

# Generate Submission
sub_save_path = os.path.join(drive_out_path, "submission.csv")
create_kaggle_submission(model, testloader, sub_save_path)

Using device: cuda

Preloading Train images into RAM. This takes ~1 minute...


Loading: 100%|██████████| 50000/50000 [00:14<00:00, 3473.82it/s]


Preloading complete!

Preloading Test images into RAM. This takes ~1 minute...


Loading: 100%|██████████| 2000/2000 [00:00<00:00, 4408.67it/s]


Preloading complete!

Total Trainable Parameters: 4,731,978



/tmp/ipykernel_688/2959804116.py:189: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/120:   0%|          | 0/196 [00:00<?, ?it/s]/tmp/ipykernel_688/2959804116.py:201: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 120/120: 100%|██████████| 196/196 [00:35<00:00,  5.57it/s, Loss=0.362, Acc=95.03%]



Model saved safely to /content/drive/MyDrive/dl-fa26-resnet-project/efficient_resnet_final.pth

Generating predictions for hidden test set...


Predicting: 100%|██████████| 8/8 [00:03<00:00,  2.49it/s]

Saved 2000 predictions to /content/drive/MyDrive/dl-fa26-resnet-project/submission.csv. Ready to submit!


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Turns on automatic hardware optimization for static image sizes
torch.backends.cudnn.benchmark = True

# ==========================================
# 1. ARCHITECTURE (<5M Params)
# ==========================================
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class CustomBasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, use_se=True):
        super(CustomBasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.use_se = use_se
        if self.use_se:
            self.se = SEBlock(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.use_se:
            out = self.se(out)
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class EfficientResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(EfficientResNet, self).__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # ADDED DROPOUT TO PREVENT OVERFITTING
        self.dropout = nn.Dropout(0.3)
        self.linear = nn.Linear(256 * block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.dropout(out) # Apply Dropout before final prediction
        out = self.linear(out)
        return out

def get_efficient_resnet():
    return EfficientResNet(CustomBasicBlock, [4, 4, 3])


# ==========================================
# 2. DATASET READER (RAM PRELOADER)
# ==========================================
class KaggleCIFARDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None, is_test=False, val_split=0.0, is_val=False):
        self.df = pd.read_csv(csv_file)
        self.df.columns = [str(c).strip().lower() for c in self.df.columns]

        # --- TRAIN/VAL SPLIT LOGIC ---
        if not is_test and val_split > 0:
            # Shuffle deterministically so train and val don't overlap
            self.df = self.df.sample(frac=1, random_state=42).reset_index(drop=True)
            split_idx = int(len(self.df) * (1 - val_split))
            if is_val:
                self.df = self.df.iloc[split_idx:] # Last 10% for Validation
            else:
                self.df = self.df.iloc[:split_idx] # First 90% for Training
        # -----------------------------

        self.transform = transform
        self.is_test = is_test
        self.images = []
        self.targets = []

        dataset_type = "Test" if is_test else ("Validation" if is_val else "Train")
        print(f"\nPreloading {dataset_type} images into RAM. ({len(self.df)} images)")

        for idx in tqdm(range(len(self.df)), desc="Loading"):
            row = self.df.iloc[idx]

            raw_id = str(row['id']) if 'id' in self.df.columns else str(row.iloc[0])
            clean_id_number = ''.join(filter(str.isdigit, raw_id))

            img_name = f"test_{clean_id_number.zfill(4)}.png" if self.is_test else f"train_{clean_id_number.zfill(5)}.png"
            img_path = os.path.join(img_dir, img_name)

            if not self.is_test:
                target = int(row['label']) if 'label' in self.df.columns else int(row.iloc[1])
            else:
                target = row['id'] if 'id' in self.df.columns else raw_id

            image = Image.open(img_path).convert("RGB")
            self.images.append(image)
            self.targets.append(target)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image = self.images[idx]
        target = self.targets[idx]
        if self.transform:
            image = self.transform(image)
        return (target, image) if self.is_test else (image, target)


# Transforms with AutoAugment for maximum accuracy
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Directories (Uses the BASE_DIR found automatically in Cell 1)
train_csv_path = os.path.join(BASE_DIR, "train_labels.csv")
train_img_dir = os.path.join(BASE_DIR, "train_images")
test_csv_path = os.path.join(BASE_DIR, "sample_submission.csv")
test_img_dir = os.path.join(BASE_DIR, "test_images")

# Create the 3 Splits
trainset = KaggleCIFARDataset(train_csv_path, train_img_dir, transform_train, is_test=False, val_split=0.1, is_val=False)
valset = KaggleCIFARDataset(train_csv_path, train_img_dir, transform_test, is_test=False, val_split=0.1, is_val=True)
testset = KaggleCIFARDataset(test_csv_path, test_img_dir, transform_test, is_test=True)

trainloader = DataLoader(trainset, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
valloader = DataLoader(valset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
testloader = DataLoader(testset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

# ==========================================
# 3. TRAINING LOOP (WITH CHECKPOINTING)
# ==========================================
def train_and_validate(model, trainloader, valloader, epochs, save_path):
    model = model.to(device)

    # Label Smoothing prevents the model from being "too confident" (Great for overfitting)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    optimizer = optim.SGD(model.parameters(), lr=0.2, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler()

    best_val_acc = 0.0

    for epoch in range(epochs):
        # -- 1. TRAINING PHASE --
        model.train()
        train_loss, correct, total = 0, 0, 0
        pbar = tqdm(enumerate(trainloader), total=len(trainloader), desc=f"Epoch {epoch+1}/{epochs} [TRAIN]")

        for batch_idx, (inputs, targets) in pbar:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()

            with torch.cuda.amp.autocast():
                outputs = model(inputs)
                loss = criterion(outputs, targets)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            pbar.set_postfix({"Loss": f"{train_loss/(batch_idx+1):.3f}", "Acc": f"{100.0 * correct / total:.2f}%"})

        scheduler.step()

        # -- 2. VALIDATION PHASE --
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for inputs, targets in valloader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += targets.size(0)
                val_correct += predicted.eq(targets).sum().item()

        val_acc = 100.0 * val_correct / val_total
        print(f"--> Val Loss: {val_loss/len(valloader):.4f} | Val Acc: {val_acc:.2f}%")

        # -- 3. BEST CHECKPOINTING --
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f"*** New Best Model Saved! ({best_val_acc:.2f}%) ***")


# ==========================================
# 4. GENERATE KAGGLE SUBMISSION
# ==========================================
def create_kaggle_submission(model, dataloader, filename="submission.csv"):
    model.eval()
    print("\nGenerating predictions for hidden test set...")

    results = []
    with torch.no_grad():
        for ids, inputs in tqdm(dataloader, desc="Predicting"):
            inputs = inputs.to(device)
            outputs = model(inputs)
            predictions = outputs.argmax(dim=1).cpu().numpy()

            for img_id, pred in zip(ids, predictions):
                if isinstance(img_id, torch.Tensor):
                    img_id = img_id.item()
                results.append({"id": img_id, "label": pred})

    df = pd.DataFrame(results)
    df.to_csv(filename, index=False)
    print(f"Saved {len(df)} predictions to {filename}. Ready to submit!")


# ==========================================
# 5. RUN EVERYTHING
# ==========================================
model = get_efficient_resnet()
drive_out_path = "/content/drive/MyDrive/dl-fa26-resnet-project/"
model_save_path = os.path.join(drive_out_path, "efficient_resnet_best.pth")

# Train and save the best checkpoints
EPOCHS = 120
train_and_validate(model, trainloader, valloader, epochs=EPOCHS, save_path=model_save_path)

# LOAD THE BEST WEIGHTS BEFORE SUBMITTING
print("\nLoading best model weights for final submission...")
model.load_state_dict(torch.load(model_save_path))

# Generate Submission
sub_save_path = os.path.join(drive_out_path, "submission.csv")
create_kaggle_submission(model, testloader, sub_save_path)

Using device: cuda

Preloading Train images into RAM. (45000 images)


Loading: 100%|██████████| 45000/45000 [00:15<00:00, 2890.68it/s]



Preloading Validation images into RAM. (5000 images)


Loading: 100%|██████████| 5000/5000 [00:01<00:00, 2869.18it/s]



Preloading Test images into RAM. (2000 images)


Loading: 100%|██████████| 2000/2000 [00:00<00:00, 4085.16it/s]
/tmp/ipykernel_2517/2711264670.py:202: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/120 [TRAIN]:   0%|          | 0/176 [00:00<?, ?it/s]/tmp/ipykernel_2517/2711264670.py:216: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/120 [TRAIN]: 100%|██████████| 176/176 [00:42<00:00,  4.14it/s, Loss=1.977, Acc=29.55%]


... (237 outputs left out)

Epoch 120/120 [TRAIN]: 100%|██████████| 176/176 [00:38<00:00,  4.59it/s, Loss=0.734, Acc=95.02%]


--> Val Loss: 0.6806 | Val Acc: 95.24%
*** New Best Model Saved! (95.24%) ***

Loading best model weights for final submission...

Generating predictions for hidden test set...


Predicting: 100%|██████████| 8/8 [00:02<00:00,  3.79it/s]


Saved 2000 predictions to /content/drive/MyDrive/dl-fa26-resnet-project/submission.csv. Ready to submit!
